# Ingeniería del Dato: BASE 2 - María Duque Laredo

## Carga de librerías

In [1]:
import pandas as pd
from pathlib import Path

import pandas as pd 
from functools import reduce

import seaborn as sns

import matplotlib.pyplot as plt

import seaborn as sns

## Dataset 1: Gasto en educación por CCAA

In [38]:
# Carga del dataset 1: Gasto en educación por CCAA (INE)

gasto_educacion = pd.read_csv("../raw/gasto_hogares_educacion.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos las columnas
gasto_educacion = gasto_educacion.rename(columns={
    "Comunidades y ciudades autónomas": "ccaa",
    "Tipo de gasto": "tipo_gasto",
    "Indicador": "indicador",
    "Total": "gasto_reglado_medio"
})

# De la variable 'ccaa' eliminamos la fila de 'Total nacional'

gasto_educacion = gasto_educacion[
    ~gasto_educacion['ccaa'].str.contains("Total", case=False)
]

# Eliminamos la variable 'indicador' porque no nos aporta nada

gasto_educacion = gasto_educacion.drop(columns=["indicador"])

# Normalizamos los nombres de las CCAA

reemplazos = {
    "Asturias, Principado de": "Asturias",
    "Balears, Illes": "Islas Baleares",
    "Castilla - La Mancha": "Castilla La Mancha",
    "Madrid, Comunidad de": "Madrid",
    "Murcia, Región de": "Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Andalucía": "Andalucia",
    "Aragón": "Aragon",
    "Castilla y León": "Castilla y Leon",
    "País Vasco": "Pais Vasco"

}

gasto_educacion["ccaa"] = gasto_educacion["ccaa"].replace(reemplazos)

# Eliminamos la variable 'tipo_gasto' porque su valor es siempre el mismo 'Gasto en Servicios Educativos reglados'

gasto_educacion = gasto_educacion.drop(columns=['tipo_gasto'])

# Como la variable gasto_reglado_medio esta tipo string y no numérico, lo convertimos                                   

gasto_educacion['gasto_reglado_medio'] = pd.to_numeric(
    gasto_educacion['gasto_reglado_medio']
    .astype(str)
    .str.replace('.', '', regex=False),
    errors='coerce'
)

# Ordenamos el dataset para que se parezca al resto y sea más fácil el merge

gasto_educacion = gasto_educacion.sort_values(by='ccaa').reset_index(drop=True)

# Verificamos que no existen duplicados

print(
    "Duplicados:",
    gasto_educacion.duplicated(
        ["ccaa", "gasto_reglado_medio"]
    ).sum()
)

# Visualizamos el dataset limpio

display(gasto_educacion)

# Guardamos el dataset limpio

gasto_educacion.to_csv(
    "../processed/gasto_hogares_educacion_clean.csv",
    index=False
)

print(gasto_educacion['ccaa'].unique())
print(gasto_educacion.head())
print(gasto_educacion.columns)
print(gasto_educacion.shape)

Duplicados: 0


,ccaa,gasto_reglado_medio
0,Andalucia,1128
1,Aragon,9700
2,Asturias,8950
3,Canarias,1446
4,Cantabria,6800
5,Castilla La Mancha,8610
6,Castilla y Leon,1121
7,Cataluña,1945
8,Ceuta,3980
9,Comunidad Valenciana,1437


<StringArray>
[           'Andalucia',               'Aragon',             'Asturias',
             'Canarias',            'Cantabria',   'Castilla La Mancha',
      'Castilla y Leon',             'Cataluña',                'Ceuta',
 'Comunidad Valenciana',          'Extremadura',              'Galicia',
       'Islas Baleares',             'La Rioja',               'Madrid',
              'Melilla',               'Murcia',              'Navarra',
           'Pais Vasco']
Length: 19, dtype: str
        ccaa  gasto_reglado_medio
0  Andalucia                 1128
1     Aragon                 9700
2   Asturias                 8950
3   Canarias                 1446
4  Cantabria                 6800
Index(['ccaa', 'gasto_reglado_medio'], dtype='str')
(19, 2)


## Dataset 2: Centros educativos por CCAA

In [73]:
# Carga del dataset 1: Gasto en educación por CCAA (INE)

n_centros_ccaa = pd.read_csv("../raw/centros_ccaa.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos columnas

n_centros_ccaa = n_centros_ccaa.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Tipo de centro": "tipo_centro",
    "Total": "n_centros"})

# Eliminamos columna 'Titularidad/financiación del centro', no es relevante para este dataset

n_centros_ccaa = n_centros_ccaa.drop(
    columns=["Titularidad/financiación del centro"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA (Comunidad Foral de) (5)": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].replace(reemplazos_ccaa)

# Convertimos la variable 'n_centros' a numérica

n_centros_ccaa['n_centros'] = pd.to_numeric(
    n_centros_ccaa['n_centros']
    .astype(str)
    .str.replace('.', '', regex=False),
    errors='coerce'
)

# Eliminamos las filas de la variable 'tipo de centro' cuyo valor sea 'TOTAL'

n_centros_ccaa = n_centros_ccaa[
    n_centros_ccaa["tipo_centro"] != "TOTAL"
]

# Normalizamos los nombres de los tipos de centro

reemplazos_tipo = {
    "Centros E. Primaria (2)": "Centro_primaria",
    "Centros ESO y/o Bachillerato y/o FP (3)": "Centro_secundaria"
}

n_centros_ccaa["tipo_centro"] = n_centros_ccaa["tipo_centro"].replace(reemplazos_tipo)

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_centros_pivot = n_centros_ccaa.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="n_centros"
).reset_index()

# Eliminamos el nombre del indice 'tipo_centro' de las columnas 

df_centros_pivot.columns.name = None

# Vamos a calcular la proporción de cada tipo de centro por ccaa
# 1. Calculamos el número total de centros considerados por comunidad autónoma

df_centros_pivot["total_centros"] = (
    df_centros_pivot["Centro_primaria"] + df_centros_pivot["Centro_secundaria"]
)

# 2. Creamos variables relativas para evitar que el modelo sesgue por tamaño
# Primaria

df_centros_pivot["pct_centro_primaria"] = (
    df_centros_pivot["Centro_primaria"] / df_centros_pivot["total_centros"]
)

# Secundaria

df_centros_pivot["pct_centro_secundaria"] = (
    df_centros_pivot["Centro_secundaria"] / df_centros_pivot["total_centros"]
)

# Nos quedamos con las variables finales más útiles para el modelo

df_centros_final = df_centros_pivot[
    ["ccaa", "pct_centro_primaria", "pct_centro_secundaria"]
].copy()

# Verificamos que no haya nulos

print(df_centros_final.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_centros_final.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_centros_final)
print(df_centros_final.info())

# Guardamos el dataset limpio y pivotado

df_centros_final.to_csv(
    "../processed/centros_ccaa_clean.csv",
    index=False
)

ccaa                     0
pct_centro_primaria      0
pct_centro_secundaria    0
dtype: int64
Duplicados: 0


,ccaa,pct_centro_primaria,pct_centro_secundaria
0,Andalucia,0.142857,0.857143
1,Aragon,0.662844,0.337156
2,Asturias,0.668712,0.331288
3,Canarias,0.706564,0.293436
4,Cantabria,0.669767,0.330233
5,Castilla La Mancha,0.710583,0.289417
6,Castilla y Leon,0.680000,0.320000
7,Cataluña,0.182296,0.817704
8,Ceuta,0.708333,0.291667
9,Comunidad Valenciana,0.174560,0.825440


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ccaa                   19 non-null     str    
 1   pct_centro_primaria    19 non-null     float64
 2   pct_centro_secundaria  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None
